# Naive Bayes

**Companion lesson:** https://ml-viz.vercel.app/courses/probabilistic-models/03-naive-bayes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## A spam filter in 25 lines

Multinomial Naive Bayes on a toy corpus — training really is just counting.

In [ ]:
spam = ["win free money now", "free viagra click now", "claim your free prize money",
        "urgent winner click here", "free money waiting claim now"]
ham  = ["meeting moved to monday", "lunch tomorrow with the team", "draft of the report attached",
        "can you review my code", "notes from the morning meeting"]

vocab = sorted(set(" ".join(spam + ham).split()))
V = len(vocab); idx = {w: i for i, w in enumerate(vocab)}

def counts(docs):
    c = np.zeros(V)
    for d in docs:
        for w in d.split(): c[idx[w]] += 1
    return c

ALPHA = 1.0   # Laplace smoothing
c_spam, c_ham = counts(spam), counts(ham)
logp_w_spam = np.log((c_spam + ALPHA) / (c_spam.sum() + ALPHA * V))
logp_w_ham  = np.log((c_ham  + ALPHA) / (c_ham.sum()  + ALPHA * V))
logprior_spam = np.log(len(spam) / (len(spam) + len(ham)))
logprior_ham  = np.log(len(ham)  / (len(spam) + len(ham)))

In [ ]:
def classify(text):
    ls, lh = logprior_spam, logprior_ham
    for w in text.split():
        if w in idx:
            ls += logp_w_spam[idx[w]]; lh += logp_w_ham[idx[w]]
    p_spam = 1 / (1 + np.exp(lh - ls))
    return p_spam

for t in ["free money meeting", "review the report", "claim free prize", "click now to win"]:
    print(f'{t!r:32}  P(spam) = {classify(t):.3f}')

## Which words carry the signal?

Each word votes with its log-likelihood ratio.

In [ ]:
llr = logp_w_spam - logp_w_ham
order = np.argsort(llr)
words = [vocab[i] for i in np.concatenate([order[:7], order[-7:]])]
vals_ = np.concatenate([llr[order[:7]], llr[order[-7:]]])

plt.figure(figsize=(8, 4.5))
plt.barh(words, vals_, color=['#14b8a6' if v < 0 else '#f43f5e' for v in vals_])
plt.xlabel('log P(w|spam) − log P(w|ham)   (votes toward spam →)')
plt.tight_layout(); plt.show()

## Why smoothing is mandatory

In [ ]:
# rebuild WITHOUT smoothing and classify a spammy email containing one ham-only word
logp_ns_spam = np.log(np.where(c_spam > 0, c_spam / c_spam.sum(), 1e-300))
text = "free money prize meeting"   # 'meeting' never appears in spam
ls = logprior_spam + sum(logp_ns_spam[idx[w]] for w in text.split())
print('unsmoothed spam log-score:', ls)
print('-> one unseen word drove the score to -inf territory,')
print('   vetoing three strong spam words. α=1 smoothing fixes this.')

**Try it:** add a third class ('newsletter'), or port this to a real dataset — scikit-learn's `fetch_20newsgroups` with `CountVectorizer` + this exact math reaches ~85% accuracy on 20 classes.